In [18]:
import os
import pandas as pd

# Directory containing the .flow files
directory = 'CTU_13/2_neris'

# List to store individual dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.flow'):
        # Read the file into a dataframe
        df = pd.read_csv(os.path.join(directory, filename))  # Adjust the reading method if necessary
        
        # Add a label column with the filename (without extension) as the label value
        #df['label'] = os.path.splitext(filename)[0]
        
        # Append the dataframe to the list
        dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=True)

combined_df.rename(columns={'Label': 'label'}, inplace=True)

# Remove rows where 'label' contains 'background'
#combined_df = combined_df[~combined_df['Label'].str.contains('background', case=False, na=False)]

# Update the 'label' column: 'Botnet' -> 1, 'Normal' -> 0
#combined_df['label'] = combined_df['label'].apply(lambda x: 1 if 'Botnet' in x else (0 if 'Normal' in x else x))

# Update the 'label' column: 'Botnet' -> 1, 'Normal' -> 0, 'Background' -> 0
combined_df['label'] = combined_df['label'].apply(lambda x: 1 if 'Botnet' in x else (0 if 'Normal' in x or 'Background' in x else x))

# Optionally reset the index if needed
combined_df.reset_index(drop=True, inplace=True)

# Display the combined dataframe
print(combined_df)


         TotBytes  SrcBytes  DstBytes  SrcGap  DstGap  sMeanPktSz  dMeanPktSz  \
0             508       208       300     0.0     0.0   69.333336        75.0   
1             184       122        62     0.0     0.0   61.000000        62.0   
2             184       122        62     0.0     0.0   61.000000        62.0   
3             184       122        62     0.0     0.0   61.000000        62.0   
4             184       122        62     0.0     0.0   61.000000        62.0   
...           ...       ...       ...     ...     ...         ...         ...   
1808117        62        62         0     0.0     0.0   62.000000         0.0   
1808118        79        79         0     NaN     NaN   79.000000         0.0   
1808119        77        77         0     NaN     NaN   77.000000         0.0   
1808120        66        66         0     0.0     0.0   66.000000         0.0   
1808121        78        78         0     NaN     NaN   78.000000         0.0   

         sMaxPktSz  dMaxPkt

In [19]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Handle missing values for non-numeric data
        if not non_numeric_cols.empty:
            imputer_non_numeric = SimpleImputer(strategy='most_frequent')
            df_non_numeric = pd.DataFrame(imputer_non_numeric.fit_transform(df[non_numeric_cols]), columns=non_numeric_cols)
            # Convert non-numeric features to one-hot encoding
            encoder = OneHotEncoder(drop='first')
            encoded = encoder.fit_transform(df_non_numeric)
            encoded_df = pd.DataFrame(encoded.toarray(), columns=encoder.get_feature_names_out(non_numeric_cols))
        else:
            encoded_df = pd.DataFrame()  # Empty DataFrame for encoded non-numeric data if no non-numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric, encoded_df], axis=1)
        '''
        features = [
            'State__FA', 'State__A', 'pRetran', 'Max,sMeanPktSz', 'SrcRetra', 'PCRatio', 'State_PA_', 'State_PA_A',
            'SrcWin,SrcLoss', 'DstRate', 'SrcLoad', 'TcpOpt_MwsS  T', 'Load', 'DstLoad', 'TcpRtt', 'Flgs_ e g      ',
            'State_A_PA', 'Flgs_ e d      ', 'Sum', 'AckDat', 'dTtl', 'State__PA', 'Min', 'pLoss', 'DstLoss', 'State_S_',
            'Cause_Status', 'State_FA_', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'Flgs_ e s      ',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'State_S_SA', 'DstPkts', 'Flgs_ e *      ', 'Mean',
            'SrcBytes', 'State__SA', 'TotBytes', 'Cause_Start', 'dMeanPktSz', 'State_A_A', 'DstRetra', 'SynAck'
        ]
        '''
        features = [
            'AckDat', 'TcpRtt', 'SynAck', 'IdleTime', 'SrcBytes', 
            'DstWin', 'TcpOpt_MwsS  T', 'Min', 'Dur', 'pLoss',
            'Flgs_ e s      ', 'State_S_', 'TotPkts', 'DstPkts'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        scaler = MinMaxScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        return df_scaled
    else:
        return pd.DataFrame()
    
# Step 1: Separate the label column
X = combined_df.drop(columns=['label'])
y = combined_df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(len(pre_df.columns))
pre_df['label']

15


0          0
1          0
2          0
3          0
4          0
          ..
1808117    0
1808118    0
1808119    0
1808120    0
1808121    0
Name: label, Length: 1808122, dtype: int64

Data loaded, shape: (129832, 15)


In [30]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from scipy.stats import binom
from sklearn.metrics import accuracy_score

# Phase 1: Network Data Retrieval
class NetworkDataRetrieval:
    def __init__(self, df, window_size):
        self.data = df
        self.window_size = window_size

    def load_data(self):
        print(f"Data loaded, shape: {self.data.shape}")

    def sliding_window(self):
        windows = []
        for i in range(0, len(self.data) - self.window_size + 1):
            window = self.data.iloc[i:i + self.window_size]
            windows.append(window)
        return windows

# Phase 2: Canopy Clustering and Cluster Updates
class CanopyClustering:
    def __init__(self, n_clusters=5, threshold=0.25):
        self.n_clusters = n_clusters
        self.threshold = threshold
        self.train_clusters = None
        self.training_data = None
        self.kmeans = None  # Store the trained KMeans model

    def apply_kmeans(self, data):
        self.kmeans = KMeans(n_clusters=self.n_clusters)
        clusters = self.kmeans.fit_predict(data)
        return clusters

    def train_initial_clusters(self, windows):
        # Apply KMeans clustering on the first three windows
        combined_data = pd.concat(windows[:3], axis=0).dropna()
        clusters = self.apply_kmeans(combined_data.iloc[:, :-1])  # Fit KMeans once on initial data
        self.training_data = combined_data
        print(f"Initial training clusters created with {len(np.unique(clusters))} clusters.")
        return clusters

    def predict_new_data_cluster(self, new_data):
        if self.kmeans is not None:
            # Use the trained KMeans model to predict clusters for new data
            predicted_clusters = self.kmeans.predict(new_data.dropna().iloc[:, :-1])
            print(f"Predicted clusters for new data: {np.unique(predicted_clusters)}")
            return predicted_clusters
        else:
            print("KMeans model not trained yet.")
            return None

    def update_training_clusters(self, new_data):
        # Add new data to training clusters
        self.training_data = pd.concat([self.training_data, new_data], axis=0)
        print(f"Training data updated with {new_data.shape[0]} new samples.")
        print(self.training_data.shape[0])

# Phase 3: Model Prediction and Drift Detection
class ModelPredictionDriftDetection:
    def __init__(self):
        self.model = SVC(kernel='linear')

    def train_model(self, X_train, y_train):
        if len(np.unique(y_train)) > 1:  # Ensure more than one class
            self.model.fit(X_train, y_train)
            print("SVM Model trained.")
        else:
            print("Training skipped: only one class present.")

    def predict(self, X_test):
        return self.model.predict(X_test)

    def error_rate_based_drift_detection(self, y_true, y_pred, confidence=0.95):
        errors = sum(y_pred != y_true)
        p_error = errors / len(y_true)
        conf_interval = binom.interval(confidence, len(y_true), p_error)
        if errors > conf_interval[1]:
            print("Drift detected based on error rate.")

# Phase 4: Concept Drift Adaptation
class ConceptDriftAdaptation:
    def __init__(self, window_size=50, threshold=0.9, n_clusters=5, drift_confidence=0.95):
        self.window_size = window_size
        self.threshold = threshold
        self.n_clusters = n_clusters
        self.drift_confidence = drift_confidence
        self.canopy = CanopyClustering(n_clusters=n_clusters)
        self.model_drift = ModelPredictionDriftDetection()
        self.last_accuracy = None
        self.drift_warning_count = 0  # Count for consecutive warnings
        self.drift_detected = False

    def process_windows(self, windows):
        # Train initial model on the first 3 windows
        combined_data = pd.concat(windows[:3], axis=0).dropna()
        clusters = self.canopy.train_initial_clusters(windows[:3])

        X_train, y_train = combined_data.iloc[:, :-1], combined_data.iloc[:, -1]
        self.model_drift.train_model(X_train, y_train)

        # Monitor subsequent windows
        for i in range(3, len(windows) - 2):
            SWi_next = windows[i]
            SWi_next_next = windows[i + 1]

            # Phase 3: Predict and detect drift
            X_test, y_test = SWi_next.iloc[:, :-1], SWi_next.iloc[:, -1]
            predictions = self.model_drift.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)

            # Drift detection based on accuracy and warnings
            if self.last_accuracy is not None and accuracy <= self.threshold:
                print(f"Warning: Accuracy below threshold in window {i}. Accuracy: {accuracy}")
                self.drift_warning_count += 1

                # Check if we have 3 consecutive warnings (drift detection)
                if self.drift_warning_count >= 3:
                    print("Drift detected after 3 consecutive warnings.")
                    self.drift_detected = True

                    # Phase 2: Predict clusters of new data using trained KMeans
                    combined_next_data = pd.concat([SWi_next, SWi_next_next], axis=0).dropna()
                    predicted_clusters = self.canopy.predict_new_data_cluster(combined_next_data)
                    print(predicted_clusters)

                    # Use drifted data to update corresponding training clusters
                    self.canopy.update_training_clusters(combined_next_data)

                    # Retrain model with updated clusters
                    X_train_new, y_train_new = combined_next_data.iloc[:, :-1], combined_next_data.iloc[:, -1]
                    self.model_drift.train_model(X_train_new, y_train_new)

                    predictions = self.model_drift.predict(SWi_next_next.iloc[:, :-1])
                    accuracy = accuracy_score(SWi_next_next.iloc[:, -1], predictions)
                    print(f"Model retrained. New accuracy: {accuracy}")

                    # Reset warning count after drift adaptation
                    self.drift_warning_count = 0

            else:
                self.drift_warning_count = 0  # Reset warning count if no drift

            # Update historical accuracy
            self.last_accuracy = accuracy

# Example usage
# Assuming pre_df is defined in a previous cell
phase1 = NetworkDataRetrieval(pre_df, window_size=500)
phase1.load_data()
windows = phase1.sliding_window()

# Phase 4
drift_adaptation = ConceptDriftAdaptation(window_size=500, threshold=0.9, n_clusters=5, drift_confidence=0.95)
drift_adaptation.process_windows(windows)


Data loaded, shape: (1808122, 15)
Initial training clusters created with 5 clusters.
SVM Model trained.
Drift detected after 3 consecutive warnings.
Predicted clusters for new data: [0 1 2 3 4]
[1 2 1 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 2 1 1
 1 1 1 1 1 1 1 0 0 1 2 2 1 1 2 2 1 2 2 2 2 2 2 2 2 2 1 1 1 1 2 1 2 2 1 1 1
 1 1 1 1 1 1 2 1 1 2 2 2 2 1 1 1 1 1 2 2 1 1 1 1 2 1 1 1 1 1 2 2 2 2 1 2 2
 1 2 1 2 2 2 1 1 1 1 1 1 2 1 2 2 2 1 1 2 1 2 2 2 1 2 2 2 1 2 1 0 0 0 0 0 2
 1 3 1 3 3 3 1 1 3 0 1 1 4 3 1 3 3 3 4 3 2 3 3 3 4 1 1 3 3 3 3 2 1 4 3 1 1
 1 0 1 1 1 3 0 1 1 3 1 1 1 1 1 0 1 1 1 0 1 1 1 3 3 3 1 1 1 1 3 4 1 1 1 1 1
 1 3 3 3 3 1 1 1 0 3 3 1 1 1 3 4 1 3 3 1 1 3 1 0 1 1 1 3 3 3 3 1 3 3 3 1 0
 3 3 1 3 3 1 0 1 3 1 1 1 3 3 0 3 3 3 3 3 1 1 1 1 3 3 0 1 1 1 2 1 0 3 3 3 3
 0 3 1 1 1 1 1 1 3 3 3 3 1 3 1 3 3 1 4 3 3 3 1 1 3 3 0 1 0 1 3 0 0 3 3 3 1
 1 0 1 1 1 3 1 3 3 1 0 1 0 3 1 1 2 1 3 3 1 1 0 1 0 3 3 0 1 3 3 1 3 3 3 1 3
 0 0 2 1 1 0 1 0 1 1 1 1 1 1 1 3 3 0 0 3 1 1 1 1 1 1 1 0

KeyboardInterrupt: 